# [실습2] Langchain으로 시장조사 문서 기반 챗봇 만들기 - PDF

## 실습 목표
---
[실습1] RAG를 위한 Vector Score, Retriever 에서 학습한 내용을 바탕으로 LangChain을 활용해서 입력된 문서를 요약해서 Context로 활용하는 챗봇을 개발합니다.

## 실습 목차
---

1. **시장조사 문서 벡터화:** RAG 챗봇에서 활용하기 위해 시장조사 파일을 읽어서 벡터화하는 과정을 실습합니다.

2. **RAG 체인 구성:** 이전 실습에서 구성한 미니 RAG 체인을 응용해서 간단한 시장 조사 문서 기반 RAG 체인을 구성합니다.

3. **챗봇 구현 및 사용:** 구성한 RAG 체인을 활용해서 시장조사 문서 기반 챗봇을 구현하고 사용해봅니다.

## 실습 개요
---
RAG 체인을 활용해서 시장조사 문서 기반 챗봇을 구현하고 사용해봅니다.

## 0. 환경 설정
- 필요한 라이브러리를 불러옵니다.

In [1]:
from langchain.document_loaders import PyPDFLoader
from langchain_community.chat_models import ChatOllama
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# from langchain_openai import ChatOpenAI, OpenAIEmbeddings
# OPENAI_API_KEY= ""

- Ollama를 통해 Mistral 7B 모델을 불러옵니다.

In [ ]:
#!ollama pull mistral:7b

## 1. 시장조사 문서 벡터화
- RAG 챗봇에서 활용하기 위해 시장조사 파일을 읽어서 벡터화하는 과정을 실습합니다.

먼저, mistral:7b 모델을 사용하는 ChatOllama 객체와 OllamaEmbeddings 객체를 생성합니다.

In [2]:
llm = ChatOllama(model="mistral:7b")
embeddings = OllamaEmbeddings(model="mistral:7b")

# llm_openai = ChatOpenAI(model='gpt-4o-mini', openai_api_key=OPENAI_API_KEY)
# embeddings_openai = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

다음으로, 시장조사 PDF 문서를 불러와서 벡터화 해보겠습니다.
- 한국소비자원의 2022년 키오스크(무인정보단말기) 이용 실태조사 보고서를 활용했습니다
  - https://www.kca.go.kr/smartconsumer/sub.do?menukey=7301&mode=view&no=1003409523&page=2&cate=00000057
- 이 실태조사 보고서는 2022년 키오스크의 사용자 경험, 접근성, 후속 조치에 대해 논의하는 보고서입니다. 
- 이를 활용해서 키오스크를 어떻게 세일즈 할 수 있을지 아이디어를 제공하는 챗봇을 만들어야 하는 상황이라고 가정해 봅시다.

먼저, LangChain의 `PyPDFLoader`를 활용해서 시장조사 보고서의 텍스트를 추출하고, 페이지 별로 `Document`를 생성하여 저장합니다.

In [3]:
doc_path = "docs/키오스크(무인정보단말기) 이용실태 조사.pdf"
loader = PyPDFLoader(doc_path)
docs = loader.load()

생성된 Document의 수를 확인해봅시다.

In [4]:
print(len(docs))

59


다음으로, 각 Document의 길이를 확인해봅시다.

In [7]:
docs[10]

Document(metadata={'source': 'docs/키오스크(무인정보단말기) 이용실태 조사.pdf', 'page': 10}, page_content='키오스크 (무인정보단말기 )이용 실태조사\n11 시장조사국 시장감시팀ㅇ 최근 한국소비자원의 「고령소비자 비대면 거래 실태조사 (’20.7.)」결과 ,\n고령소비자가 ‘주문단계 복잡 ’,‘조작의 어려움 ’등의 이유로 키오스크 \n이용에 부담12)을 느끼는 것으로 나타났으며 ,\nㅇ 언론 등을 통해 장애인의 키오스크 접근성 문제 해결 필요성도 지속 \n제기되고 있음 .\n[표2-2-2] 고령자·장애인의 키오스크 이용 불편 관련 언론 보도13)\nㅇ (고령자) “(…)한 프랜차이즈 카페. 중절모를 쓴 노인 네 명이 키오스크 앞에서 우왕좌왕 \n한참을 헤맸다. 손님이 몰릴 시간대라 직원들은 카운터에서 주문을 받지 않고 음료를 \n만드는 데만 집중하고 있었다. 이들은 뒤에서 기다리던 기자의 도움으로 겨우 주문을 \n마쳤다. 아메리카노 네 잔을 시키는 데 걸린 시간은 10분이 넘었다.(…)”\nㅇ (장애인) “(…)시각장애인에게도 키오스크는 ‘유리 장벽’이다. 점자 패드나 음성 \n안내 기능이 있는 ‘배리어 프리’ 키오스크가 거의 존재하지 않는 터라 무인화된 \n매장에서 장애인들은 혼자 힘으로는 음료수 한 잔도 주문할 수 없다. 휠체어를 탄 \n사람들에게도 키오스크 모니터 화면이 너무 높아 이용할 수가 없다.(…)”\nㅇ 한국지능정보사회진흥원이 매년 정보취약계층을 대상으로 「디지털정보격차 \n실태조사」를 실시하고 있으나 ,이는 컴퓨터․모바일 기기 중심이므로 현재 \n디지털 약자층의 키오스크 접근․활용도에 대한 정확한 통계는 없음 .\n□(이슈 2:시스템 비표준화에 따른 혼란 )키오스크의 화면 구성 및 조작 \n방법 등(UI/ UX*)이 표준화되어 있지 않아 ,디지털 약자층뿐만 아니라 \n비장애인 및 청년층에서도 키오스크 이용 중 불편을 겪는 사례가 다수 발생\nㅇ 특히 탄소중립 실천을 위해 적극적으로 텀블러를 사

In [8]:
doc_len = [len(doc.page_content) for doc in docs]
print(doc_len)

[86, 6679, 6011, 5915, 1201, 908, 676, 1289, 821, 1154, 1439, 447, 1031, 1178, 514, 1083, 968, 1119, 1006, 1094, 978, 916, 1230, 862, 680, 1251, 1433, 1290, 729, 1170, 1011, 598, 733, 966, 934, 1195, 514, 1210, 777, 635, 651, 771, 837, 397, 953, 877, 548, 1022, 1198, 1183, 1230, 838, 533, 1255, 1231, 1894, 777, 798, 662]


1천자 미만의 문서도 있지만, 6천자가 넘는 문서도 있는 것을 확인할 수 있습니다. 이대로 그냥 사용할 경우, Context가 너무 길어져 오히려 성능이 낮아질 수도 있습니다.

우선은 이대로 RAG 체인을 구성해 봅시다.

## 2. RAG 체인 구성
RAG 체인을 구성하기 위해 `Document`를 `OllamaEmbeddings`를 활용해 벡터로 변환하고, FAISS DB를 활용하여 저장합니다.
- 변환 및 저장 과정은 약 3분 정도 소요됩니다.

In [9]:
vectorstore = FAISS.from_documents(
    docs,
    embedding=embeddings
)

# vectorstore_openai = FAISS.from_documents(
#     docs,
#     embedding=embeddings_openai
# )

In [10]:
db_retriever = vectorstore.as_retriever()

# db_retriever_openai = vectorstore_openai.as_retriever()


In [21]:
#db_retriever.invoke("키오스크 관련 설문조사 결과를 알려줘")  # 55page
db_retriever.invoke("연령대별 키오스크 이용 이유를 알려줘")  # 15page

# db_retriever_openai.invoke("키오스크 설문조사의 조사 기간 및 담당자를 알려줘")

[Document(metadata={'source': 'docs/키오스크(무인정보단말기) 이용실태 조사.pdf', 'page': 15}, page_content='키오스크 (무인정보단말기 )이용 실태조사\n16 시장조사국 시장감시팀4. 무인정보단말기 접근성 지침(KS X 9211 :2022)(산업표준심의회 , 2022.2.25. 개정)\n□(주요 내용)무인정보단말기를 장애인도 접근 가능하도록 설계하는 방법을 \n제시한 것으로 ,이 표준에 따라 설계된 무인정보단말기는 비장애인뿐만 \n아니라 다양한 장애를 지닌 사용자나 고령자도 쉽게 이용할 수 있음 .\nㅇ △무인정보단말기 주변 환경 (휠체어 사용자 접근성 ),△하드웨어 접근성\n(시‧\n청각 ,지체장애인 등 접근성 ),△온-스크린 (소프트웨어 )접근성 ,△편의 \n제공 요구사항 등을 규정하여 모든 소비자가 키오스크를 쉽게 이용할 수\n있도록 설계하는 방안을 제시\nㅇ 다만 ,의무 규정은 아니므로 강제력이 없음 .\n5. 정책 동향19)\n□(국회)김예지 의원실은 보건복지부의 시행령에 대하여 ‘과도하게 구체적으로 \n명시된 시행령의 내용이 새로운 기술의 진입을 차단할 뿐만 아니라 ,\n경제적이면서 실효성 있는 제품의 상용화에도 걸림돌이 될 수 있다’고 지적 \nㅇ 또한 2023년 1월 이전에 설치된 키오스크는 2026년까지 3년 유예기간을 \n명시하고 있는 점과 관련하여 실효성 의문 제기20)\n□(보건복지부 )공청회 (’22.6.27.)이후 「장애인차별금지 및 권리구제 등에 관한 \n법률 시행령」초안에 대한 의견을 수렴하였으며 ,현재 시행령 최종안을 \n마련 중\nㅇ 특히 단계적 적용의 기간과 기술발전을 위한 폭넓은 시행령 내용을 검토 중이며 ,\n시행령에 구체적인 규정을 열거하는 대신 「장애인‧\n고령자 등의 정보 접근 및 \n이용 편의 증진을 위한 고시」의 내용을 준용하도록 하는 방안도 검토중\n6. 해외사례21)\n□(유럽 )「European Accessibility Act」법령을 통해 유럽 국가 내 제품

이전 실습에서 구성한 미니 RAG Chain과 비슷하게 Chain을 구성해 봅시다.
- 지난 실습과 달리 이번 챗봇의 역할은 마케터를 위한 챗봇으로 고정했으므로, 역할을 별도로 인자로 전달할 필요가 없습니다.
- `RunnablePassthrough()`는 Chain의 이전 구성 요소에서 전달된 값을 그대로 전달하는 역할을 수행합니다.

In [11]:
def get_retrieved_text(docs):
    result = "\n".join([doc.page_content for doc in docs])
    return result

def init_chain():
    messages_with_contexts = [
        ("system", "당신은 마케터를 위한 친절한 지원 챗봇입니다. 사용자가 입력하는 정보를 바탕으로 질문에 답하세요."),
        ("human", "정보: {context}.\n{question}."),
    ]

    prompt_with_context = ChatPromptTemplate.from_messages(messages_with_contexts)

    # 체인 구성
    # context에는 질문과 가장 비슷한 문서를 반환하는 db_retriever에 get_retrieved_text를 적용한 chain의 결과값이 전달됩니다.
    qa_chain = (
        {"context": db_retriever | get_retrieved_text, "question": RunnablePassthrough()}
        | prompt_with_context
        | llm
        | StrOutputParser()
    )
    
    return qa_chain

In [12]:
qa_chain = init_chain()

Chain 구성이 완료되었습니다.

## 3. 챗봇 구현 및 사용
- 구성한 RAG 체인을 활용해서 시장조사 문서 기반 챗봇을 구현하고 사용해봅니다.

방금 구현한 RAG Chain을 사용해서 시장조사 문서 기반 챗봇을 구현해볼 것입니다. 

그 전에, 별도로 RAG 기능을 추가하지 않은 LLM과 답변의 퀄리티를 비교해 봅시다.

In [13]:
messages_with_variables = [
    ("system", "당신은 마케터를 위한 친절한 지원 챗봇입니다."),
    ("human", "{question}."),
]
prompt = ChatPromptTemplate.from_messages(messages_with_variables)
parser = StrOutputParser()
chain = prompt | llm | parser

In [16]:
print(chain.invoke("키오스크 관련 설문조사 결과를 알려줘"))

 안녕하세요! 당신은 마케팅 분야의 동료로 어서오세요. 최근에 우리 회사가 키오스크 관련 설문조사를 진행했는데, 다음은 일부 결과입니다.

1. 주로 이용하시는 앱/서비스: Spotify(47%), Netflix(38%), Amazon(35%)
2. 키오스크를 통해 가장 많이 구입한 품목: 음료(60%), 간편식(45%), 주말용 물품(30%)
3. 키오스크의 장점은? (다중선택)
   - 24시간 운영(78%)
   - 신속한 결제(76%)
   - 고객센터와 직접적인 통신이 어려워서(19%)
   - 신세계의 체험(14%)
4. 키오스크의 단점은? (다중선택)
   - 고객센터와 직접적인 통신이 어려워서(58%)
   - 신속한 결제가 어려울 때(23%)
   - 물건이 많아질 경우 찾기가 불편해진다(10%)
   - 책임의 감을 들게된다(8%)
5. 키오스크를 더욱 좋아하시겠습니까? (예: 72%, 아니요: 28%)
6. 추가로 소중한 피드백이 있으신가요? (예: 50%)
    - "키오스크의 UI/UX가 더 개선되어야 합니다."
    - "키오스크에서도 고객센터와 직접적인 통신이 가능하게 해주세요."
    - "키오스크의 품목을 더 많이 제공하여, 장점만큼 단점도 줄일 수 있으시면 좋겠습니다."

이러한 설문조사결과를 토대로 마케팅 전략을 수립할 계획입니다. 지속적인 고객 섬겨주시는 것에 감사합니다! 더 나은 서비스를 제공하기 위해 노력하겠습니다.


In [23]:
print(qa_chain.invoke("연령대별 키오스크 이용 이유를 알려줘"))
#연령대별 키오스크 이용 이유를 알려줘

 아래의 정보에서는 29개 업종의 피해를 경험한 응답자들을 분석한 결과, '외식업'에서 가장 많은 피해가 발생하였으며, 모든 연령대가 키오스크를 가장 많이 이용하기 때문으로 추정된다. 또한, 업종별 키오스크 관련 피해 경험 여부와 유형에 대한 분석이 수행되었으며, 일부 피해 유형으로는 '강제 이용', '취소 불가', '변경 불가', '주문 실수' 및 '상품 미제공'이 있다.

연령대별 키오스크를 사용하는 이유는 아래와 같습니다.

1. 편리성: 시간 및 영업시간 제한 없이 24시간 365일 서비스를 받을 수 있으므로, 쉽고 빠르게 상품과 서비스를 구입할 수 있다.
2. 안정감: 직원 없이 자가서비스를 제공하여, 직원의 부족이나 장애에 따른 피해가 발생하는 것을 방지할 수 있다.
3. 개인정보 유지: 키오스크를 통해 일련번호와 같은 일시적이고 익명화된 정보만 입력하여, 정보 유출을 방지할 수 있다.
4. 신용카드 거래: 기존의 현금 및 체크 외에 신용카드나 영수증을 사용하여 편리한 거래가 가능하고, 잠재적인 피해를 줄일 수 있다.
5. 디지털화: 디지털 시장의 확산과 함께 모바일 및 인터넷을 통한 거래가 증대하고, 이를 활용하여 키오스크를 통해 상품과 서비스를 구입할 수 있다.

따라서, 키오스크는 편리성, 안정감, 개인정보 유지, 신용카드 거래, 디지털화와 같은 장점으로 인해 많은 사람들이 키오스크를 통한 서비스 요청에 매력을 느껴 이용하고 있는 것으로 보입니다.


일반 체인은 아무런 출처가 없는 답변을 생성한 반면, RAG 기능을 추가한 챗봇은 데이터를 기반으로 상대적으로 정확한 답변을 하는 것을 확인할 수 있습니다. 

이제 챗봇을 한번 사용해 봅시다.

In [ ]:
qa_chain = init_chain()
while True:
    question = input("질문을 입력해주세요 (종료를 원하시면 '종료'를 입력해주세요.): ")
    if question == "종료":
        break
    else:
        result = qa_chain.invoke(question)
        print(result)

저희는 이전 챕터에서 구현한 챗봇이 가지고 있는 문제점 중 '문서나 데이터 기반 추론이 불가능하다.'를 완화했습니다.

또한, 지금 구성한 챗봇은 UI가 없고 단순 표준 입출력 만을 사용합니다. 5챕터에서 Streamlit을 활용해 ChatGPT와 비슷한 웹 챗봇 어플리케이션을 제작해 볼 것입니다.